# Whale Instance Segmentation — YOLOv8 on Thermal Grayscale Images

This notebook documents the full pipeline for training and evaluating a YOLOv8 segmentation model on thermal drone images of whales.

**Pipeline overview:**
1. Dataset preparation — RGB → Grayscale conversion
2. Baseline training (YOLOv8s)
3. Experiment 1 — RGB vs Grayscale
4. Experiment 2 — Inference threshold search (conf / IoU)
5. Experiment 3 — Model architecture comparison (YOLOv8s vs YOLO26s)
6. Hyperparameter tuning (Ray Tune / Ultralytics)
7. Final evaluation — Best tuned model on test set

---
## 0. Imports & Global Configuration

All paths, constants and thresholds are defined here so they never need to be changed deeper in the notebook.

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import yaml
from pathlib import Path

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

# ── Dataset paths ─────────────────────────────────────────────────────────────
DATASET_ROOT_RGB  = "./17_03_dataset"            # Original RGB dataset
DATASET_ROOT_GRAY = "./17_03_dataset_grayscale"  # Converted grayscale dataset
DATASET_YAML      = f"{DATASET_ROOT_GRAY}/data.yaml"

# ── Model weights ─────────────────────────────────────────────────────────────
MODEL_RGB_PATH      = "./runs/segment/train/weights/best.pt"              # Baseline trained on fake-RGB
MODEL_GRAY_V8S_PATH = "./runs/segment/train_grayscale_v8s/weights/best.pt" # Baseline trained on grayscale
MODEL_GRAY_26S_PATH = "./runs/segment/train_grayscale_26s/weights/best.pt" # YOLO26s trained on grayscale
MODEL_TUNED_PATH    = "./runs/segment/train_bestparam/weights/best.pt"     # Final tuned model

# ── Inference thresholds (validated via Experiment 2) ─────────────────────────
CONF_THRESHOLD = 0.25  # Confidence threshold
IOU_THRESHOLD  = 0.30  # IoU threshold — intentionally permissive for partial whale detections

# ── Test set size (verified programmatically in Section 1) ───────────────────
N_TEST_INSTANCES = 95  # Number of annotated instances in the test split

---
## 1. Dataset Preparation

### The RGB → Grayscale issue

The thermal drone images are inherently black-and-white (single-channel), but were saved as 3-channel RGB files by duplicating the same intensity value into R, G and B.

**Why this hurts training:**
- The model wastes capacity learning from redundant colour channels.
- Data augmentation (HSV jitter, colour noise) adds artefacts that don't exist in real thermal imagery, confusing the model.

**Fix:** Convert every image to true 1-channel grayscale before training.

In [ ]:
# Convert all RGB images in the dataset to true 1-channel grayscale.
# Images are overwritten in-place — run this only once on a copy of the dataset.

splits = ["train", "valid", "test"]
total_converted = 0

print("🔄 Starting RGB → Grayscale conversion...")

for split in splits:
    img_dir = Path(DATASET_ROOT_RGB) / split / "images"

    if not img_dir.exists():
        print(f"   ⚠️  Skipped '{split}': directory not found at {img_dir}")
        continue

    files = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.jpeg")) + list(img_dir.glob("*.png"))

    if not files:
        print(f"   ⚠️  Skipped '{split}': no images found.")
        continue

    print(f"   Processing '{split}' ({len(files)} images)...")

    for img_path in files:
        try:
            img = Image.open(img_path)
            if img.mode == "RGB":
                img.convert("L").save(img_path)  # L = 8-bit grayscale
                total_converted += 1
        except Exception as e:
            print(f"   ❌ Error on {img_path.name}: {e}")

print(f"\n✅ Done — {total_converted} images converted to true 1-channel grayscale.")

In [ ]:
# Verify dataset split sizes and confirm images are truly grayscale.

print("📂 Dataset split sizes:")
for split in ["train", "valid", "test"]:
    img_dir = Path(DATASET_ROOT_GRAY) / split / "images"
    images = list(img_dir.rglob("*.jpg")) + list(img_dir.rglob("*.png"))
    print(f"   {split:6s}: {len(images)} images")

# Spot-check one image to confirm channel count
sample_img_path = next((Path(DATASET_ROOT_GRAY) / "train" / "images").rglob("*.jpg"))
img = Image.open(sample_img_path)
channels = len(img.getbands())
print(f"\n🔍 Sample image: {sample_img_path.name}")
print(f"   Size: {img.width} x {img.height} px")
print(f"   Channels: {channels} ({'✅ Grayscale' if channels == 1 else '❌ Still RGB — rerun conversion'})")

---
## 2. Model Training

Two architectures are trained on the grayscale dataset:
- **YOLOv8s-seg** — the standard small segmentation model (baseline)
- **YOLO26s-seg** — a more recent architecture to compare against

> **Note:** YOLO automatically resizes images to the nearest multiple of 32.  
> `imgsz=432` → actual training resolution is **448px** (432 rounded up to 448).  
> This is expected behaviour and does not affect results.

In [ ]:
# ── Train YOLOv8s (baseline) ──────────────────────────────────────────────────
model_v8s = YOLO("yolov8s-seg.pt")

model_v8s.train(
    data=DATASET_YAML,
    epochs=100,
    imgsz=432,
    batch=16,
    workers=4,
    name="train_grayscale_v8s"
)

In [ ]:
# ── Train YOLO26s ─────────────────────────────────────────────────────────────
model_26s = YOLO("yolo26s-seg.pt")

model_26s.train(
    data=DATASET_YAML,
    epochs=100,
    imgsz=432,
    batch=16,
    workers=4,
    name="train_grayscale_26s"
)

---
## 3. Evaluation Helper

A single reusable function used across all experiments.  
All comparisons use `split="test"` (data never seen during training) with fixed thresholds defined in Section 0.

In [ ]:
def evaluate_model(name: str, model_path: str) -> dict | None:
    """
    Evaluate a YOLO segmentation model on the test set.

    Parameters
    ----------
    name : str
        Display name used in the comparison table.
    model_path : str
        Path to the model weights (.pt file).

    Returns
    -------
    dict with mAP, Recall, Precision and estimated False Negatives,
    or None if evaluation fails.
    """
    try:
        model = YOLO(model_path)
        metrics = model.val(
            data=DATASET_YAML,
            split="test",
            conf=CONF_THRESHOLD,
            iou=IOU_THRESHOLD,
            verbose=False,
            plots=False
        )
        recall = metrics.seg.r.mean()
        return {
            "Model":                 name,
            "mAP50-95 (Box)":        f"{metrics.box.map:.4f}",
            "mAP50-95 (Mask)":       f"{metrics.seg.map:.4f}",
            "Recall (Mask)":         f"{recall:.4f}",
            "Precision (Mask)":      f"{metrics.seg.p.mean():.4f}",
            "False Negatives (Est)": int(N_TEST_INSTANCES * (1 - recall))
        }
    except Exception as e:
        print(f"❌ Error evaluating '{name}': {e}")
        return None


def print_comparison(results: list[dict], title: str = "HEAD-TO-HEAD COMPARISON") -> pd.DataFrame:
    """Display a formatted comparison table and declare a winner by mAP50-95 (Mask)."""
    df = pd.DataFrame(results)
    print("\n" + "=" * 80)
    print(f"🏆  {title}")
    print("=" * 80)
    print(df.to_string(index=False))
    print("=" * 80)

    scores = df.set_index("Model")["mAP50-95 (Mask)"].astype(float)
    best_model = scores.idxmax()
    diff = scores.max() - scores.min()
    print(f"\n✅ Best model: {best_model}  (margin: +{diff:.4f} mAP)")
    return df

---
## 4. Experiment 1 — RGB vs Grayscale

**Hypothesis:** Training on true 1-channel grayscale images should outperform training on fake 3-channel RGB images because colour augmentations (HSV jitter) introduce noise that doesn't exist in thermal imagery.

Both models use the same YOLOv8s architecture and are evaluated on the same grayscale test set.

In [ ]:
results_exp1 = []

print("📊 Evaluating RGB baseline...")
res = evaluate_model("YOLOv8s — RGB (fake 3-channel)", MODEL_RGB_PATH)
if res: results_exp1.append(res)

print("📊 Evaluating Grayscale model...")
res = evaluate_model("YOLOv8s — Grayscale (true 1-channel)", MODEL_GRAY_V8S_PATH)
if res: results_exp1.append(res)

df_exp1 = print_comparison(results_exp1, "Experiment 1 — RGB vs Grayscale (Test Set)")

---
## 5. Experiment 2 — Inference Threshold Search (conf / IoU)

**Goal:** Find the conf and IoU thresholds that maximise Recall (minimise missed whales) without sacrificing too much Precision.

**Context:** Whale detection from thermal drone imagery tolerates a higher false-positive rate better than false negatives — a missed whale is worse than a spurious detection.

The best weights from Experiment 1 (grayscale YOLOv8s) are used here.
These thresholds are **post-processing only** — they don't require retraining.

In [ ]:
model_gray = YOLO(MODEL_GRAY_V8S_PATH)

# Grid of threshold combinations to evaluate
threshold_grid = [
    {"conf": 0.50, "iou": 0.30, "note": "High confidence (strict)"},
    {"conf": 0.25, "iou": 0.30, "note": "Lower confidence"},
    {"conf": 0.25, "iou": 0.25, "note": "Lower confidence + lower IoU"},
    {"conf": 0.15, "iou": 0.20, "note": "Maximum recall (more false positives expected)"},
]

results_exp2 = []
print("📊 Scanning threshold combinations on test set...\n")

for s in threshold_grid:
    metrics = model_gray.val(
        data=DATASET_YAML,
        split="test",
        conf=s["conf"],
        iou=s["iou"],
        verbose=False,
        plots=False
    )
    recall = metrics.seg.r.mean()
    fn = int(N_TEST_INSTANCES * (1 - recall))
    results_exp2.append({
        "Setting":         s["note"],
        "conf":            s["conf"],
        "iou":             s["iou"],
        "Recall":          f"{recall:.4f}",
        "Precision":       f"{metrics.seg.p.mean():.4f}",
        "False Negatives": fn
    })
    print(f"   {s['note']:45s} → Recall={recall:.4f} | FN={fn}")

df_exp2 = pd.DataFrame(results_exp2)
print("\n" + "=" * 80)
print("📊 Experiment 2 — Threshold Search Results")
print("=" * 80)
print(df_exp2.to_string(index=False))
print("\n→ Selected thresholds: conf=0.25, iou=0.30 (best recall / precision balance)")

---
## 6. Experiment 3 — Model Architecture: YOLOv8s vs YOLO26s

**Hypothesis:** YOLO26s, a more recent architecture, may extract richer features from the grayscale thermal images despite having a similar parameter count.

Both models are trained identically (same dataset, epochs, image size) and evaluated with the thresholds selected in Experiment 2.

In [ ]:
results_exp3 = []

print("📊 Evaluating YOLOv8s...")
res = evaluate_model("YOLOv8s", MODEL_GRAY_V8S_PATH)
if res: results_exp3.append(res)

print("📊 Evaluating YOLO26s...")
res = evaluate_model("YOLO26s", MODEL_GRAY_26S_PATH)
if res: results_exp3.append(res)

df_exp3 = print_comparison(results_exp3, "Experiment 3 — YOLOv8s vs YOLO26s (Test Set)")

---
## 7. Hyperparameter Tuning

**Goal:** Automatically search for training hyperparameters (learning rate, augmentation settings, etc.) that improve on the YOLOv8s baseline.

**Method:** Ultralytics `model.tune()` uses Ray Tune with the **ASHA scheduler** — an early-stopping algorithm that allocates more epochs to promising trials and kills poor ones early. This is more efficient than a full grid search.

**Important:** The test set is not touched during tuning. Only the validation split is used to score each trial.

In [ ]:
# ── Launch hyperparameter search ──────────────────────────────────────────────
# Each trial trains for `epochs` epochs — kept short to explore quickly.
# Results are saved to runs/segment/tune/

model_to_tune = YOLO("yolov8s-seg.pt")

model_to_tune.tune(
    data=DATASET_YAML,
    epochs=30,        # Short per trial — exploration phase
    iterations=50,    # Number of hyperparameter combinations to try
    optimizer="AdamW",
    plots=True,       # Generates tune_fitness.png and scatter plots
    save=True,
    val=True,
)

In [ ]:
# ── Analyse tune results ──────────────────────────────────────────────────────
# fitness = 0.1 * mAP50(box+mask) + 0.9 * mAP50-95(box+mask) — bounded [0, 2] for segmentation

df_tune = pd.read_csv("./runs/segment/tune/tune_results.csv")

# Top 10 trials by fitness
print("Top 10 trials by fitness:")
print(df_tune.sort_values("fitness", ascending=False)[
    ["fitness", "lr0", "mosaic", "degrees", "flipud", "scale"]
].head(10).to_string(index=False))

# Fitness evolution — did we converge?
df_tune["best_so_far"] = df_tune["fitness"].cummax()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(df_tune["fitness"], "o", alpha=0.4, label="Trial fitness")
axes[0].plot(df_tune["best_so_far"], "-", color="crimson", linewidth=2, label="Best so far")
axes[0].set_xlabel("Trial"); axes[0].set_ylabel("Fitness")
axes[0].set_title("Fitness evolution across trials")
axes[0].legend()

# Per-parameter correlation with fitness
correlations = df_tune.drop(columns=["fitness"]).corrwith(df_tune["fitness"]).sort_values()
correlations.plot(kind="barh", ax=axes[1], color="steelblue")
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Hyperparameter correlation with fitness")
axes[1].set_xlabel("Pearson correlation")

plt.tight_layout()
plt.show()

In [ ]:
# ── Final training with best hyperparameters ──────────────────────────────────
# Load the best hyperparameters found by the tuner and train for more epochs.

with open("./runs/segment/tune/best_hyperparameters.yaml") as f:
    best_params = yaml.safe_load(f)

print("Best hyperparameters:")
for k, v in best_params.items():
    print(f"   {k}: {v}")

model_final = YOLO("yolov8s-seg.pt")

model_final.train(
    data=DATASET_YAML,
    epochs=150,        # Full training run — longer than tune trials
    imgsz=432,
    name="train_bestparam",
    **best_params      # Inject tuned hyperparameters
)

---
## 8. Final Evaluation — Best Model on Test Set

The test set is used **only here**, once, to report the final performance of the tuned model.
Using it earlier (e.g., during threshold search) would bias the reported scores.

We compare:
- **Baseline** — YOLOv8s with default hyperparameters
- **Tuned** — YOLOv8s with best hyperparameters from Ray Tune

In [ ]:
# ── Baseline vs Tuned — Test Set ──────────────────────────────────────────────
results_final = []

print("📊 Evaluating Baseline (YOLOv8s, default params)...")
res = evaluate_model("Baseline — YOLOv8s (default)", MODEL_GRAY_V8S_PATH)
if res: results_final.append(res)

print("📊 Evaluating Tuned model (YOLOv8s, best params)...")
res = evaluate_model("Tuned — YOLOv8s (Ray Tune)", MODEL_TUNED_PATH)
if res: results_final.append(res)

df_final = print_comparison(results_final, "Final Comparison — Baseline vs Tuned (Test Set)")

In [ ]:
# ── Detailed validation report — Tuned model ─────────────────────────────────
# plots=True generates PR curves and prediction visualisations in runs/segment/val/

model_tuned = YOLO(MODEL_TUNED_PATH)

metrics = model_tuned.val(
    data=DATASET_YAML,
    split="val",
    conf=CONF_THRESHOLD,
    iou=IOU_THRESHOLD,
    verbose=True,
    plots=True
)

print("\n" + "=" * 60)
print("📊 VALIDATION RESULTS — Tuned Model (best.pt)")
print("=" * 60)
print(f"mAP50        (Box):  {metrics.box.map50:.4f}")
print(f"mAP50-95     (Box):  {metrics.box.map:.4f}")
print(f"mAP50        (Mask): {metrics.seg.map50:.4f}")
print(f"mAP50-95     (Mask): {metrics.seg.map:.4f}")
print(f"Recall       (Mask): {metrics.seg.r.mean():.4f}")
print(f"Precision    (Mask): {metrics.seg.p.mean():.4f}")
print("=" * 60)